# Лабораторная работа 1
## 2D диод с резким p-n переходом

Считаем плоский кремниевый диод. Сверху анод, снизу катод,
посередине резкий переход: сверху бор, снизу мышьяк.

Четыре этапа как в методичке: геометрия и примесь, сетка,
решение при 4 В на аноде, картинки.

In [1]:
import sys, os
ROOT = os.path.abspath(".")
while not os.path.isdir(os.path.join(ROOT, "tools")) and ROOT != "/":
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, "tools"))

os.makedirs(os.path.join(ROOT, "Lab1_diode_2D", "ise"), exist_ok=True)
os.chdir(os.path.join(ROOT, "Lab1_diode_2D", "ise"))

import xp, put, dfise, viz
import numpy as np

WORK = r"C:\tcad\LAB1"     # папка внутри Windows
print(os.getcwd())

/Users/phoenix/Documents/MIET/labs_modelling/work/Lab1


## Где считаем

Ниже — вывод самих программ ISE. По нему видно версию Windows,
имя машины, версию решателя с копирайтом разработчика.

In [2]:
print(xp.run(["ver", "hostname", "echo %ISEROOT% %ISERELEASE%"]))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>ver

Microsoft Windows XP [Версия 5.1.2600]

C:\Documents and Settings\Легошин>hostname
win-xp

C:\Documents and Settings\Легошин>echo %ISEROOT% %ISERELEASE%
C:\ISE 7.0

C:\Documents and Settings\Легошин>


In [3]:
# решатель сам печатает версию, разработчика, ОС и машину
print(xp.run([r"C:\ISE\bin\dessis -h"])[:1600])

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>C:\ISE\bin\dessis -h
*******************************************************************
                      DESSIS - version 7.0.6
              (compiled Wed Feb 28 16:33:09  2001)

   Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
                        Zurich, Switzerland
*******************************************************************

	Running on machine with the following configuration:
	Host Name: WIN-XP
	Operating System: WindowsNT rel. 4.0 ver. 0
	Machine Type: ix86
	Date: Tue Sep 29 09:40:29 2026  (╠юёъютёъюх тЁхь (чшьр))

Usage: dessis [ options ] [ filename[.cmd] ]
Options are:
-c                    write current and acextraction files in old format
-d         

In [5]:
# ---- ВАРИАНТ ----
VARIANT = 2    # номер
d       = 0.8    # толщина диода по Oy, мкм
NB      = 1.4e+17    # бор, 1/см3
NAs     = 1.0e+17    # мышьяк, 1/см3
# -----------------
assert None not in (VARIANT, d, NB, NAs), "не заполнен вариант"

# в методичке d = 2 мкм. всё по Oy делим на k, по Ox не трогаем
k = 2.0 / d
h = d / 2.0
print(VARIANT, d, k)

2 0.8 2.5


## 1. Структура

Прямоугольник кремния и два контакта. Анод не на всю ширину, поэтому ток растекается.

In [6]:
bnd = f"""Silicon "SiW"
{{
\trectangle [(-1.5,{-h}) (1.5,{h})]
}}

Contact "Anode"
{{
\tline[(-0.5,{-h}) (0.5,{-h})]
}}

Contact "Cathode"
{{
\tline[(-1.5,{h}) (1.5,{h})]
}}
"""
open("sh1.bnd","w").write(bnd)
print(bnd)

Silicon "SiW"
{
	rectangle [(-1.5,-0.4) (1.5,0.4)]
}

Contact "Anode"
{
	line[(-0.5,-0.4) (0.5,-0.4)]
}

Contact "Cathode"
{
	line[(-1.5,0.4) (1.5,0.4)]
}



## 2. Сетка и примесь

Три области сгущения вместо двух, и это не произвол — так сетка совпадает
с эталонной из `Variants_1-2_2023.doc` узел в узел. Проверялось по картинкам:
считались линии отдельно внутри окна второй сетки и снаружи.

- `Mesh1` — общая, по Oy потолок 0.5/k. Даёт крупную ячейку у катода.
- `Mesh3` — верхняя половина во всю ширину, вдвое мельче. Без неё верх выходит
  таким же грубым, как низ, а в эталоне он мельче.
- `Mesh2` — окно у анода, ещё вдвое мельче и по Ox тоже.

`MaxElementSize` — не размер ячейки, а потолок: генератор режет пополам, пока
ячейка не станет меньше потолка. Поэтому `0.2` на окне шириной 2 мкм даёт 0.125,
а не 0.2, и числа тут не круглые не случайно.

In [ ]:
cmd = f"""Title "Sharp Diode"

Definitions {{
    Refinement "Mesh1"
    {{
      MaxElementSize = (0.5 {0.5/k:g})
      MinElementSize = (0.1 {0.1/k:g})
      RefineFunction = MaxTransDiff(Variable = "DopingConcentration", Value = 1)
    }}
    Refinement "Mesh3"
    {{
      MaxElementSize = (0.5 {0.25/k:g})
      MinElementSize = (0.1 {0.1/k:g})
    }}
    Refinement "Mesh2"
    {{
      MaxElementSize = (0.25 {0.2/k:g})
      MinElementSize = (0.05 {0.01/k:g})
    }}
    Constant "BW"  {{ Species = "BoronActiveConcentration"   Value = {NB} }}
    Constant "AsW" {{ Species = "ArsenicActiveConcentration" Value = {NAs} }}
}}

Placements {{
    Refinement "Mesh1" {{ Reference = "Mesh1"
      RefineWindow = rectangle [( -1.5 {-h} ) , ( 1.5 {h} )] }}
    Refinement "Mesh3" {{ Reference = "Mesh3"
      RefineWindow = rectangle [( -1.5 {-h} ) , ( 1.5 0 )] }}
    Refinement "Mesh2" {{ Reference = "Mesh2"
      RefineWindow = rectangle [( -1.0 {-h} ) , ( 1.0 {-h/2} )] }}
    Constant "BW"  {{ Reference = "BW"
      EvaluateWindow {{ Element = rectangle [( -1.5 {-h} ) , ( 1.5 0 )] DecayLength = 0 }} }}
    Constant "AsW" {{ Reference = "AsW"
      EvaluateWindow {{ Element = rectangle [( -1.5 0 ) , ( 1.5 {h} )] DecayLength = 0 }} }}
}}
"""
open("sh1.cmd","w").write(cmd)
print(cmd)

## 3. Файл решателя

Напряжение на аноде поднимаем с 0 до 4 В шагами по 0.01.

In [8]:
des = """File {
\tGrid = "sh1_msh.grd"
\tDoping = "sh1_msh.dat"
\tPlot = "sh1_des.dat"
\tCurrent = "sh1_des.plt"
\tOutput = "sh1_des.log"
}

Electrode {
\t{ Name="Anode"   Voltage=0.0 }
\t{ Name="Cathode" Voltage=0.0 }
}

Physics {
\tMobility( DopingDep HighFieldsat Enormal )
\tEffectiveIntrinsicDensity(BandGapNarrowing (OldSlotboom))
}

Plot {
\teDensity hDensity eCurrent hCurrent
\tPotential SpaceCharge ElectricField
\teMobility hMobility eVelocity hVelocity
\tDoping DonorConcentration AcceptorConcentration
\tCurrent/Vector
}

Math { Extrapolate Derivatives RelErrControl NewDiscretization }

Solve {
\tPoisson
\tCoupled { Poisson Electron Hole}
\tQuasistationary ( MaxStep=0.01 Goal { Name="Anode" Voltage=4 } )
\t{ Coupled { Poisson Electron Hole} }
}
"""
open("sh1_des.cmd","w").write(des)

# линия разреза: сверху вниз по центру. четыре числа — x,y начала и x,y конца
open("cut.txt","w").write(f"0.0\n{-h}\n0.0\n{h}\n")
print(des)

File {
	Grid = "sh1_msh.grd"
	Doping = "sh1_msh.dat"
	Plot = "sh1_des.dat"
	Current = "sh1_des.plt"
	Output = "sh1_des.log"
}

Electrode {
	{ Name="Anode"   Voltage=0.0 }
	{ Name="Cathode" Voltage=0.0 }
}

Physics {
	Mobility( DopingDep HighFieldsat Enormal )
	EffectiveIntrinsicDensity(BandGapNarrowing (OldSlotboom))
}

Plot {
	eDensity hDensity eCurrent hCurrent
	Potential SpaceCharge ElectricField
	eMobility hMobility eVelocity hVelocity
	Doping DonorConcentration AcceptorConcentration
	Current/Vector
}

Math { Extrapolate Derivatives RelErrControl NewDiscretization }

Solve {
	Poisson
	Coupled { Poisson Electron Hole}
	Quasistationary ( MaxStep=0.01 Goal { Name="Anode" Voltage=4 } )
	{ Coupled { Poisson Electron Hole} }
}



`xp` — самописный, `tools/xp.py`. ISE стоит в виртуалке с Windows XP,
ноутбук на маке. `xp` ходит туда по telnet и гоняет команды.
Ждёт приглашение `>` перед каждой следующей, иначе долгие расчёты обрываются.

- `xp.run([...])` — выполнить команды, вернуть вывод
- `xp.fetch(путь, куда)` — забрать файл из Windows
- `put.commands_for(файл, путь)` — команды заливки файла внутрь

In [9]:
# заливаем файлы в Windows
cmds = [f"mkdir {WORK}"]
for f in ("sh1.bnd", "sh1.cmd", "sh1_des.cmd", "cut.txt",):
    cmds += put.commands_for(f, f"{WORK}\\{f}")
xp.run(cmds, total=2400)
print("залито")

залито


## Сетка

`mesh -t` строит сетку из треугольников. На выходе `sh1_msh.grd` и `sh1_msh.dat`.

In [10]:
# строим сетку
print(xp.run([f"cd /d {WORK}", r"C:\ISE\bin\mesh -t sh1"], total=1800, idle_limit=300))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB1

C:\tcad\LAB1>C:\ISE\bin\mesh -t sh1
> # ISE TCAD Software Release 7.0
> # Version Mdraw 7.0.11
> # Version Mesh 7.0.11
> # Compiled Fri Feb  9 13:16:00  2001 on TERM101
> # Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
> # ISE TCAD Software Release 7.0
> # Version Mdraw 7.0.11
> # Version Mesh 7.0.11
> # Compiled Fri Feb  9 13:16:00  2001 on TERM101
> # Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
> Mesh version 7.0.11
> (Compiled Fri Feb  9 13:15:57  2001)
> (Compiled on TERM101)
> Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG, Zurich, Switzerland
> Running on: WindowsNT WIN-XP 4.0 0 ix86

> Process ID: 1268
>
> Flags used: -

## Решение

`dessis` решает Пуассона и непрерывность. Около минуты.

In [11]:
# решаем
print(xp.run([f"cd /d {WORK}", r"C:\ISE\bin\dessis sh1_des.cmd"], total=3600, idle_limit=600))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB1

C:\tcad\LAB1>C:\ISE\bin\dessis sh1_des.cmd
*******************************************************************
                      DESSIS - version 7.0.6
              (compiled Wed Feb 28 16:33:09  2001)

   Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
                        Zurich, Switzerland
*******************************************************************

	Running on machine with the following configuration:
	Host Name: WIN-XP
	Operating System: WindowsNT rel. 4.0 ver. 0
	Machine Type: ix86
	Date: Tue Sep 29 09:42:06 2026  (╠юёъютёъюх тЁхь (чшьр))

Input file: sh1_des.cmd

Output file: sh1_des.log

no ACExtract file

PMIPath file: C:\ISE\tcad\7.0\

## Разрез

`dfisetools` берёт координаты из `cut.txt`, поэтому в конце `< cut.txt`.

In [12]:
# делаем разрез
print(xp.run([f"cd /d {WORK}", r"C:\ISE\bin\dfisetools -plot sh1_msh.grd sh1_des.dat < cut.txt"], total=900, idle_limit=300))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB1

C:\tcad\LAB1>C:\ISE\bin\dfisetools -plot sh1_msh.grd sh1_des.dat < cut.txt

1st point of line ?
    x =     y = 2nd point of line ?
    x =     y =
C:\tcad\LAB1>


## Картинки

Рисуем не «как получится», а под эталон: цвета, толщины линий и оси сняты
пиксельно с картинок в `Variants_1-2_2023.doc`. Как именно — в конце блокнота.

In [13]:
# забираем результаты на мак
for f in ("sh1_msh.grd", "sh1_msh.dat", "sh1_des.dat", "sh1_des.plt", "sh1_des_plt.plt",):
    xp.fetch(f"{WORK}\\{f}", "" + f)
    print(f, os.path.getsize("" + f), "байт")

sh1_msh.grd 16658 байт
sh1_msh.dat 5761 байт
sh1_des.dat 91349 байт
sh1_des.plt 36196 байт
sh1_des_plt.plt 5343 байт


In [14]:
g   = dfise.read_grid("sh1_msh.grd")
dat = dfise.read_dataset("sh1_des.dat", g)
print(len(g["vertices"]), "узлов")
print(sorted(dat))

214 узлов
['AcceptorConcentration', 'DonorConcentration', 'DopingConcentration', 'ElectricField', 'ElectrostaticPotential', 'LatticeTemperature', 'SpaceCharge', 'TotalCurrentDensity-Vector', 'eCurrentDensity', 'eDensity', 'eMobility', 'eVelocity', 'hCurrentDensity', 'hDensity', 'hMobility', 'hVelocity']


In [ ]:
import ise_style as ise, compare

OUT = os.path.join(ROOT, "Lab1_diode_2D", "results")
os.makedirs(OUT, exist_ok=True)
dop = dat["DopingConcentration"]

ise.field  (g, dat["SpaceCharge"],            OUT + "/z1_заряд.png",
            mesh=True, log=True, doping=dop)
ise.contour(g, dat["ElectrostaticPotential"], OUT + "/z1_потенциал.png", doping=dop)

cut = viz.read_plt("sh1_des_plt.plt")
ise.xy([("SpaceCharge", cut["Distance"], cut["SpaceCharge"], "k")],
       OUT + "/z1_разрез.png")
print(sorted(os.listdir(OUT)))

### Заряд и сетка

Три панели: эталон из методички, наша отрисовка, вывод самого picasso.
Третья появляется, только если картинка лежит в `picasso/` в корне репозитория — почему, написано
в конце.

In [ ]:
compare.trio(compare.ref(VARIANT, "z1_заряд"),
             OUT + "/z1_заряд.png",
             compare.pic("z1_заряд"));

### Потенциал, линии уровня

In [ ]:
compare.trio(compare.ref(VARIANT, "z1_потенциал"),
             OUT + "/z1_потенциал.png",
             compare.pic("z1_потенциал"));

### Заряд в сечении от середины анода до середины катода

In [ ]:
compare.trio(compare.ref(VARIANT, "z1_разрез"),
             OUT + "/z1_разрез.png",
             compare.pic("z1_разрез"), width=6.0);

### ВАХ

In [ ]:
iv = viz.read_plt("sh1_des.plt")
V  = np.asarray(iv["Anode OuterVoltage"]); I = np.asarray(iv["Anode TotalCurrent"])
o  = np.argsort(V)
ise.xy([("TotalCurrent", V[o], I[o], "k")], OUT + "/z1_вах.png",
       size=(314, 374), legend=False)
print("ток при 4 В: %.5f А" % I[o][-1])

In [ ]:
from IPython.display import Image, display
display(Image(OUT + "/z1_вах.png"))

## Как это нарисовано, и что отсюда видно про picasso

Отрисовка своя — `tools/ise_style.py`. Писалась не «похоже», а под замер:
числа снимались пиксельно с эталонных картинок, вшитых в `Variants_1-2_2023.doc`
(в старом `.doc` картинки лежат обычными PNG, достаются поиском сигнатуры).
Мануал picasso нашёлся уже потом и почти всё подтвердил. Ниже обе колонки —
что намерено и что написано в документации.

**Сетка хранится рёбрами, а не вершинами.** В DF-ISE треугольник это три
индекса рёбер, причём отрицательный индекс значит «ребро задом наперёд».
Поэтому `dfise.triangles()` сперва собирает из рёбер замкнутое кольцо и только
потом отдаёт вершины. Поля разложены по регионам и сшиваются по `validity`.

**Контакты руками не задаются.** В файле сетки они лежат как обычные регионы,
просто материал у них `Contact`, а элементы — отрезки. Берём геометрию оттуда,
а не из `.bnd`, иначе при смене варианта разъедется.

**Палитра — ровно 31 уровень.** Это не градиент: внутри полосы цвет строго
постоянный. Крайние — `(0,42,255)` и `(153,0,0)`.

**Шкала заряда — ареасинус, и это не только про цвет.** Заряд меняет знак и
гуляет на полтора десятка порядков. На линейной шкале всё, кроме обеднённого
слоя, схлопывается в один цвет. В мануале это отдельная команда:

    device interpol <набор> linear | log | arsinh [фактор]

Ключевое слово здесь — **interpol**. Преобразуются значения *в узлах*, а
интерполяция внутри треугольника идёт уже в преобразованных координатах, и
только потом результат режется на равные полосы. Это не то же самое, что
интерполировать линейно по значению и считать границы полос по логарифму:
во втором случае картинка заметно краснее. Разница не теоретическая — на
эталонах расхождение падает с 19% до 12%, и перебор красного в нижней половине
исчезает. Ареасинус удобен тем, что у нуля ведёт себя линейно, а на хвостах
логарифмически, поэтому знак не мешает.

То есть шкала у picasso — настройка, а не свойство программы. Это же объясняет,
почему разные картинки в методичке между собой не согласованы по цвету.

**Линия перехода — объект, а не контур.** Она идеально прямая через всю ширину,
хотя эквипотенциаль под анодом обязана изгибаться. В мануале:

    model render ... junction
    model sjunction 2.0 brown

Рисуется по нулю суммарного легирования. Её легко принять за данные и подгонять
под неё шкалу — именно на этом я сначала и сбился.

**Толщины и цвета линий совпали с замером до пикселя:**

    model scontact  2.0 magenta     контакты
    model sedge     1.0 black       сетка
    model sborder   1.0 black       рамка

**Ось Y перевёрнута.** Анод стоит при `y = -h`, а на картинке он сверху —
`view up -y`.

**Диапазон цвета задаётся явно:** `model range ...`, и мануал прямо пишет, что
всё ниже минимума красится синим, всё выше максимума красным, а между ними
радуга. Отсюда и совет в той же главе — ставить границы на круглые значения.

Итого `ise_style.field()` — это переписанный `model` из picasso:
`shade face` плюс `render edge face contact junction` плюс `cmap rainbow`.

### Насколько сошлось

Сравнение считается не на глаз: берём долю площади каждого из 31 цвета у
эталона и у нас и смотрим расхождение. Сейчас 12% по заданию 1, 14% по заданию 2
и 10% по заданию 4. Полного нуля не будет — это другой запуск решателя на
другой машине, расчётные числа отличаются на единицы процентов.


### Вывод

_(где заряд какой, где обеднённая область, как выглядит ВАХ)_